In [ ]:
""" softmax """
import numpy as np

def softmax(scores):
    score_max=np.max(scores,axis=-1,keepdims=True)
    scores=scores-score_max
    scores_exp=np.exp(scores)
    return scores_exp/np.sum(scores_exp,axis=-1,keepdims=True)

scores=np.random.randn(5,4)
softmax(scores)

In [ ]:
""" 各种Loss """
import numpy as np

""" MSE Loss = (y'-y)^2 """
def MSE_Loss(y_true,y_pred):
    return np.mean(np.square(y_true-y_pred))

""" 二分类交叉熵 = - (ylog(p) + (1-y)log(1-p)) """
def Cross_entropy_loss(y_true,y_pred):
    epsilon=1e-12
    y_pred=np.clip(y_pred,epsilon,1-epsilon)
    return -np.mean(y_true*np.log(y_pred)+(1-y_true)*np.log(1-y_pred))

""" 多分类交叉熵=-sum(yi * log(pi)) """
def Cross_entropy_loss_multi_classes(y_true,y_pred):
    epsilon=1e-12
    y_pred=np.clip(y_pred,epsilon,1-epsilon)
    return -np.mean(np.sum(y_true * np.log(y_pred),axis=-1))

"""  Focal Loss=- alpha * (1-pt)^gamma *log(pt) """
def Focal_loss(y_true,y_pred,alpha=0.25,gamma=2):
    epsilon=1e-12
    y_pred=np.clip(y_pred,epsilon,1-epsilon)
    pt=np.sum(y_true * y_pred,axis=-1)
    #pt=np.clip(pt,epsilon,1-epsilon)
    loss=-alpha*(1-pt)**gamma*np.log(pt)
    return np.mean(loss)



y_pred=np.random.randn(5,4,1)
y_true=y_pred+np.random.randn(5,4,1)*0.01
Loss1=MSE_Loss(y_true,y_pred)

y_true=np.random.randint(0,2,size=(5,4,1))
y_pred=y_true+np.random.randn(5,4,1)*0.01
Loss2=Cross_entropy_loss(y_true,y_pred)

y_true=np.random.randint(0,2,size=(5,4,3))
y_pred=y_true+np.random.randn(5,4,3)*0.01
Loss3=Cross_entropy_loss_multi_classes(y_true,y_pred)

y_true=np.random.randint(0,2,size=(5,4,3))
y_pred=y_true+np.random.randn(5,4,3)*0.01
Loss4=Focal_loss(y_true,y_pred)

y_true=np.array([[0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1],
       [0, 1, 0, 0],
       [0, 0, 1, 0]])


print("MSE Loss:",Loss1)
print("Cross Entropy Loss:",Loss2)
print("Cross Entropy Loss(Multi Classes):",Loss3)
print("Focal Loss:",Loss4)

In [ ]:
import numpy as np 
from math import sqrt

class self_attention:
    def __init__(self,dim_in,dim_k,dim_v):
        self.dim_in,self.dim_k,self.dim_v=dim_in,dim_k,dim_v
        
        self.W_Q=np.random.randn(dim_in,dim_k)*0.01
        self.b_Q=np.zeros(dim_k)

        self.W_K=np.random.randn(dim_in,dim_k)*0.01
        self.b_K=np.zeros(dim_k)

        self.W_V=np.random.randn(dim_in,dim_v)*0.01
        self.b_V=np.zeros(dim_v)

        self.W_O=np.random.randn(dim_v,dim_in)*0.01
        self.b_O=np.zeros(dim_in)

        self.norm=1/sqrt(dim_k)

    def forward(self,X,mask=None):
        batch_size,seq_len,dim_in=X.shape
        # Q/K : B * L * K
        # V   : B * L * V
        Q = X @ self.W_Q +self.b_Q
        K = X @ self.W_K +self.b_K 
        V = X @ self.W_V +self.b_V
        scores= Q @ K.transpose(0,2,1) * self.norm # B * L * L
        if mask is not None:
            scores=np.where(mask==0,-1e12,scores)
        # 计算softmax
        scores_max=np.max(scores,axis=-1,keepdims=True)
        scores_exp=np.exp(scores-scores_max)
        weights=scores_exp / np.sum(scores_exp,axis=-1,keepdims=True)

        atten=weights @ V # B * L * V
        output=atten @ self.W_O - self.b_O # B * L * dim_in
        return weights,output

class MultiHeadAttention:
    def __init__(self,dim_in,dim_k,dim_v,n_head):
        self.dim_in,self.dim_k,self.dim_v,self.n_head=dim_in,dim_k,dim_v,n_head

        self.W_q=np.random.randn(dim_in,dim_k)*0.01
        self.b_q=np.zeros(dim_k)

        self.W_k=np.random.randn(dim_in,dim_k)*0.01
        self.b_k=np.zeros(dim_k)

        self.W_v=np.random.randn(dim_in,dim_v)*0.01
        self.b_v=np.zeros(dim_v)

        self.W_o=np.random.randn(dim_v,dim_in)*0.01
        self.b_o=np.zeros(dim_in)

        self.norm=(1/sqrt(dim_k//n_head))

    def forward(self,X,mask=None):
        batch_size,seq_len,dim_in=X.shape # B * L * in
        # Q/K : B * L * K
        # V   : B * L * V
        Q = X @ self.W_q +self.b_q
        K = X @ self.W_k +self.b_k 
        V = X @ self.W_v +self.b_v

        # 分头 B * n * L * k/v
        Q = Q.reshape(bacth_size,seq_len,self.n_head,-1).transpose(0,2,1,3)
        K = K.reshape(bacth_size,seq_len,self.n_head,-1).transpose(0,2,1,3)
        V = V.reshape(bacth_size,seq_len,self.n_head,-1).transpose(0,2,1,3)

        scores = Q @ K.transpose(0,1,3,2) * self.norm # B * n * L * L
        if mask is not None:
            if mask.ndim==3: # B * L * L
                mask=mask[:,None,:,:]
            scores=np.where(mask==0,-1e12,scores)
        scores_max=np.max(scores,axis=-1,keepdims=True)
        scores_exp=np.exp(scores-scores_max)
        weights=scores_exp/(np.sum(scores_exp,axis=-1,keepdims=True))
        
        atten=weights @ V # B * n * L * v
        atten=atten.transpose(0,2,1,3).reshape(batch_size,seq_len,-1) # B * L * V
        output=atten @ self.W_o +self.b_o # B * L * in
        return weights,output


if __name__ == "__main__":
    bacth_size,seq_len,dim_in=32,10,15
    X=np.random.randn(bacth_size,seq_len,dim_in)
    mask=np.random.randint(0,2,size=(bacth_size,seq_len,seq_len))
    dim_k,dim_v,n_head=12,8,4
    self_attention_np=self_attention(dim_in,dim_k,dim_v)
    MultiHeadAttention_np=MultiHeadAttention(dim_in,dim_k,dim_v,n_head)
    weights,output=MultiHeadAttention_np.forward(X,mask)
    print("Input shape:",X.shape)
    print("Attention weight shape:",weights.shape)
    print("Output shape",output.shape)
        






In [6]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
from math import sqrt

class SelfAttention(nn.Module):
    def __init__(self,dim_in,dim_k,dim_v):
        super().__init__()
        self.dim_in,self.dim_k,self.dim_v=dim_in,dim_k,dim_v
        self.W_Q=nn.Linear(dim_in,dim_k)
        self.W_K=nn.Linear(dim_in,dim_k)
        self.W_V=nn.Linear(dim_in,dim_v)
        self.W_O=nn.Linear(dim_v,dim_in)
        self.norm=1/sqrt(dim_k)

    def forward(self,X,mask=None):
        batch_size,seq_len,dim_in=X.shape
        Q,K,V=self.W_Q(X),self.W_K(X),self.W_V(X)
        scores=torch.matmul(Q,K.transpose(-2,-1))*self.norm
        if mask is not None:
            scores=scores.masked_fill(mask==0,-1e12)
        weights=F.softmax(scores,dim=-1)
        atten=torch.matmul(weights,V)
        output=self.W_O(atten)
        return weights,atten

class MultiHeadAttention(nn.Module):
    def __init__(self,dim_in,dim_k,dim_v,n_head):
        super().__init__()
        self.dim_in,self.dim_k,self.dim_v=dim_in,dim_k,dim_v
        self.n_head=n_head
        self.W_Q=nn.Linear(dim_in,dim_k)
        self.W_K=nn.Linear(dim_in,dim_k)
        self.W_V=nn.Linear(dim_in,dim_v)
        self.W_O=nn.Linear(dim_v,dim_in)
        self.norm=1/sqrt(dim_k//n_head)

    def forward(self,X,mask=None):
        batch_size,seq_len,dim_in=X.shape
        Q,K,V=self.W_Q(X),self.W_K(X),self.W_V(X)
        Q=Q.reshape(bacth_size,seq_len,self.n_head,-1).transpose(1,2)
        K=K.reshape(bacth_size,seq_len,self.n_head,-1).transpose(1,2)
        V=V.reshape(bacth_size,seq_len,self.n_head,-1).transpose(1,2)
        scores=torch.matmul(Q,K.transpose(-2,-1)) * self.norm
        if mask is not None:
            if mask.dim()==3: # B*L*L
                mask=mask.unsqueeze(1).expand(-1,self.n_head,-1,-1)
            scores=scores.masked_fill(mask==0,-1e12)
        weights=F.softmax(scores,dim=-1)
        atten=torch.matmul(weights,V)
        atten=atten.transpose(1,2).reshape(bacth_size,seq_len,-1)
        output=self.W_O(atten)
        return weights,output


if __name__ == "__main__":
    bacth_size,seq_len,dim_in=32,10,15
    X=torch.randn(bacth_size,seq_len,dim_in)
    mask=torch.randint(0,2,size=(bacth_size,seq_len,seq_len)).bool()
    dim_k,dim_v,n_head=12,8,4
    self_attention_nn=SelfAttention(dim_in,dim_k,dim_v)
    MultiHeadAttention_nn=MultiHeadAttention(dim_in,dim_k,dim_v,n_head)
    weights,output=MultiHeadAttention_nn.forward(X,mask)
    print("Input shape:",X.shape)
    print("Attention weight shape:",weights.shape)
    print("Output shape",output.shape)

Input shape: torch.Size([32, 10, 15])
Attention weight shape: torch.Size([32, 4, 10, 10])
Output shape torch.Size([32, 10, 15])
